### control via RS485

In [14]:
import minimalmodbus
import serial
import time

# --- 配置参数 (请根据实际情况修改) ---
PORT = 'COM6'   # Windows是COMx，Linux/Mac是 /dev/ttyUSB0
SLAVE_ID = 1    # 模块的站号/地址
BAUDRATE = 9600 # 波特率
target_voltage_x = 0 # unit mv
target_voltage_y = 0 # unit mv

# --- 初始化仪表对象 ---
try:
    instrument = minimalmodbus.Instrument(PORT, SLAVE_ID)
    
    # 设置串口通讯详细参数
    instrument.serial.baudrate = BAUDRATE
    instrument.serial.bytesize = 8
    instrument.serial.parity   = serial.PARITY_NONE # 常见是无校验(N)
    instrument.serial.stopbits = 1
    instrument.serial.timeout  = 1 # 秒
    
    # 调试模式：开启后会打印发送和接收的具体数据，方便排错
    instrument.debug = True 
    
    print(f"成功连接到 {PORT}，准备发送指令...")

    # --- 控制电压 ---
    # 假设场景：我们要输出 5V
    # 很多模块使用整数表示电压，例如 0-1000 对应 0-10V (即 500 = 5V)
    # 具体请看手册：是写整数(write_register) 还是写浮点数(write_float)
    
    # target_voltage =  # 目标电压
    
    # 根据手册转换数值。假设手册说：写入值 = 电压 * 100
    write_value_x = int(target_voltage_x + 30000) 
    write_value_y = int(target_voltage_y + 30000) 

    # 寄存器地址 0 (0x0000)，写入数值，小数点位数(可选)
    # functioncode=6 (写单个寄存器) 或 16 (写多个)
    # minimalmodbus 默认用 functioncode 16，有些老设备只支持 6
    instrument.write_register(registeraddress=0x0050, value=write_value_x, functioncode=6)
    instrument.write_register(registeraddress=0x0051, value=write_value_y, functioncode=6)

    print(f"指令已发送：设置输出为 {target_voltage_x}mV (写入值: {write_value_x})")
    print(f"指令已发送：设置输出为 {target_voltage_y}mV (写入值: {write_value_y})")

    # --- (可选) 读取当前电压回传 ---
    # 如果模块支持读取当前输出值，假设在寄存器 0
    # read_val = instrument.read_register(0, 1) # 读取1位小数
    # print(f"模块当前反馈电压: {read_val}")

except serial.SerialException:
    print("错误：无法打开串口，请检查USB模块是否插入或端口号是否正确。")
except minimalmodbus.NoResponseError:
    print("错误：设备无响应。请检查：")
    print("1. A/B线是否接反？")
    print("2. 波特率/地址是否匹配？")
    print("3. 模块是否已供电？")
except Exception as e:
    print(f"发生其他错误: {e}")

成功连接到 COM6，准备发送指令...
MinimalModbus debug mode. Will write to instrument (expecting 8 bytes back): 01 06 00 50 75 30 AF 5F (8 bytes)
MinimalModbus debug mode. Clearing serial buffers for port COM6
MinimalModbus debug mode. No sleep required before write. Time since previous read: 137359.00 ms, minimum silent period: 4.01 ms.
MinimalModbus debug mode. Response from instrument: 01 06 00 50 75 30 AF 5F (8 bytes), roundtrip time: 0.0 ms. Timeout for reading: 0.0 ms.

MinimalModbus debug mode. Will write to instrument (expecting 8 bytes back): 01 06 00 51 75 30 FE 9F (8 bytes)
MinimalModbus debug mode. Clearing serial buffers for port COM6
MinimalModbus debug mode. Sleeping 4.01 ms before sending. Minimum silent period: 4.01 ms, time since read: 0.00 ms.
MinimalModbus debug mode. Response from instrument: 01 06 00 51 75 30 FE 9F (8 bytes), roundtrip time: 0.0 ms. Timeout for reading: 0.0 ms.

指令已发送：设置输出为 0mV (写入值: 30000)
指令已发送：设置输出为 0mV (写入值: 30000)


### scan via RS 485

In [ ]:
import minimalmodbus
import serial
import time

# --- 1. 配置参数 ---
PORT = 'COM8'
SLAVE_ID = 1
BAUDRATE = 9600

# 扫描参数
V_START = -500    # 起始电压 (mV)
V_END   = 500  # 结束电压 (mV)
POINTS  = 10   # 每行/每列的点数 (总点数 = 50 * 50 = 2500)
OFFSET  = 30000 # 你的设备特定的偏移量

# 寄存器地址
REG_X = 0x0050
REG_Y = 0x0051

# --- 2. 初始化连接 ---
try:
    instrument = minimalmodbus.Instrument(PORT, SLAVE_ID)
    instrument.serial.baudrate = BAUDRATE
    instrument.serial.bytesize = 8
    instrument.serial.parity   = serial.PARITY_NONE
    instrument.serial.stopbits = 1
    instrument.serial.timeout  = 0.5 # 适当减小超时时间以提高效率
    
    # 先开启debug测试连接，确认连接成功后再关闭
    # instrument.debug = True 
    print(f"连接成功: {PORT} @ {BAUDRATE}")

    # --- 3. 生成扫描坐标点 ---
    # 计算步长: (2000 - 0) / (50 - 1) ≈ 40.81 mV
    # 使用列表推导式生成 0 到 2000 之间的 50 个均匀点
    scan_points = [int(V_START + i * (V_END - V_START) / (POINTS - 1)) for i in range(POINTS)]
    
    print(f"扫描范围: {V_START}mV - {V_END}mV")
    print(f"扫描步长: 约 {(V_END - V_START) / (POINTS - 1):.2f} mV")
    print(f"总点数: {POINTS} x {POINTS} = {len(scan_points) ** 2}")
    
    # ！！！重要：关闭 debug，否则扫描速度极慢且刷屏！！！
    instrument.debug = False 
    
    start_time = time.time()

    # --- 4. 开始扫描循环 ---
    # 外层循环控制 Y 轴 (行)
    for y_idx, val_y in enumerate(scan_points):
        
        # 写入 Y 轴电压 (加上偏移量)
        write_y = int(val_y + OFFSET)
        try:
            instrument.write_register(REG_Y, write_y, functioncode=6)
        except Exception as e:
            print(f"Y轴写入失败: {e}")
            continue # 如果Y失败，跳过这一行

        # 内层循环控制 X 轴 (列)
        for x_idx, val_x in enumerate(scan_points):
            
            # 写入 X 轴电压 (加上偏移量)
            write_x = int(val_x + OFFSET)
            
            try:
                instrument.write_register(REG_X, write_x, functioncode=6)
                
                # --- [在这里插入你的采集代码] ---
                # 例如：read_sensor() 或 time.sleep(0.01) 等待稳定
                # time.sleep(0.01) 
                
            except Exception as e:
                print(f"X轴写入失败 (X={val_x}, Y={val_y}): {e}")

        # 进度打印 (每完成一行打印一次，避免刷屏)
        progress = (y_idx + 1) / POINTS * 100
        print(f"进度: {progress:.1f}% | 当前 Y: {val_y} mV")

    end_time = time.time()
    duration = end_time - start_time
    
    print("-" * 30)
    print(f"扫描完成！耗时: {duration:.2f} 秒")
    print(f"平均速度: {duration / (POINTS * POINTS):.3f} 秒/点")

    # --- 5. 扫描结束后归零 (可选) ---
    print("正在归零...")
    instrument.write_register(REG_X, 0 + OFFSET, functioncode=6)
    instrument.write_register(REG_Y, 0 + OFFSET, functioncode=6)
    print("归零完毕。")

except serial.SerialException:
    print("错误：无法打开串口。")
except Exception as e:
    print(f"程序运行出错: {e}")

连接成功: COM8 @ 9600
扫描范围: -1000mV - 1000mV
扫描步长: 约 51.28 mV
总点数: 40 x 40 = 1600
进度: 2.5% | 当前 Y: -1000 mV
进度: 5.0% | 当前 Y: -948 mV
进度: 7.5% | 当前 Y: -897 mV
进度: 10.0% | 当前 Y: -846 mV
进度: 12.5% | 当前 Y: -794 mV
进度: 15.0% | 当前 Y: -743 mV
进度: 17.5% | 当前 Y: -692 mV
进度: 20.0% | 当前 Y: -641 mV
进度: 22.5% | 当前 Y: -589 mV
进度: 25.0% | 当前 Y: -538 mV
进度: 27.5% | 当前 Y: -487 mV
进度: 30.0% | 当前 Y: -435 mV
进度: 32.5% | 当前 Y: -384 mV
进度: 35.0% | 当前 Y: -333 mV
进度: 37.5% | 当前 Y: -282 mV
进度: 40.0% | 当前 Y: -230 mV
进度: 42.5% | 当前 Y: -179 mV
进度: 45.0% | 当前 Y: -128 mV
进度: 47.5% | 当前 Y: -76 mV
进度: 50.0% | 当前 Y: -25 mV
进度: 52.5% | 当前 Y: 25 mV
进度: 55.0% | 当前 Y: 76 mV
进度: 57.5% | 当前 Y: 128 mV
进度: 60.0% | 当前 Y: 179 mV
进度: 62.5% | 当前 Y: 230 mV
进度: 65.0% | 当前 Y: 282 mV
进度: 67.5% | 当前 Y: 333 mV
进度: 70.0% | 当前 Y: 384 mV
进度: 72.5% | 当前 Y: 435 mV
进度: 75.0% | 当前 Y: 487 mV
进度: 77.5% | 当前 Y: 538 mV
进度: 80.0% | 当前 Y: 589 mV
进度: 82.5% | 当前 Y: 641 mV
进度: 85.0% | 当前 Y: 692 mV
进度: 87.5% | 当前 Y: 743 mV
进度: 90.0% | 当前 Y: 794 mV
进度: 92.5

### galvo control via AFG31052

In [4]:
import pyvisa
import time

class GalvoScanner:
    def __init__(self, resource_name):
        """
        初始化 AFG31052 连接
        :param resource_name: VISA 资源地址 (例如 'USB0::0x0699::...')
        """
        self.rm = pyvisa.ResourceManager()
        try:
            self.inst = self.rm.open_resource(resource_name)
            self.inst.timeout = 5000
            print(f"已连接到仪器: {self.inst.query('*IDN?').strip()}")
        except Exception as e:
            print(f"连接失败: {e}")
            raise

    def setup_raster_scan(self, x_freq_hz, lines_per_frame, vpp_volts):
        """
        设置光栅扫描模式 (Raster Scan)
        CH1 (X轴): 快轴，锯齿波
        CH2 (Y轴): 慢轴，锯齿波
        
        :param x_freq_hz: X轴的扫描速度 (Hz)，例如 100Hz
        :param lines_per_frame: 每一帧有多少行扫描线 (决定Y轴速度)
        :param vpp_volts: 扫描幅值电压 (Vpp)，例如 4.0 表示 -2V 到 +2V
        """
        
        # 计算 Y 轴频率
        # 如果 X轴跑 100次 (100Hz)，我们需要 Y轴才跑 1次，那么就是 100行
        y_freq_hz = x_freq_hz / lines_per_frame
        
        print(f"正在配置扫描参数...")
        print(f"X轴 (CH1): {x_freq_hz} Hz, {vpp_volts} Vpp")
        print(f"Y轴 (CH2): {y_freq_hz:.4f} Hz (帧率), {lines_per_frame} 行")

        # 1. 复位仪器
        self.inst.write('*RST')
        time.sleep(1)

        # --- 配置 CH1 (X轴 - 快扫) ---
        self.inst.write('SOURce1:FUNCtion RAMP')           # 设置为斜坡波
        self.inst.write('SOURce1:FUNCtion:RAMP:SYMMetry 100') # 设置对称度100% (锯齿波)
        self.inst.write(f'SOURce1:FREQuency {x_freq_hz}')  # 设置频率
        self.inst.write(f'SOURce1:VOLTage:LEVel:IMMediate:AMPLitude {vpp_volts}')
        self.inst.write('SOURce1:VOLTage:LEVel:IMMediate:OFFSet 0')
        self.inst.write('SOURce1:PHASe 0')                 # 初始相位 0

        # --- 配置 CH2 (Y轴 - 慢扫) ---
        self.inst.write('SOURce2:FUNCtion RAMP')
        self.inst.write('SOURce2:FUNCtion:RAMP:SYMMetry 100') # 设置对称度100% (锯齿波)
        self.inst.write(f'SOURce2:FREQuency {y_freq_hz}')  # 频率 = X频率 / 行数
        self.inst.write(f'SOURce2:VOLTage:LEVel:IMMediate:AMPLitude {vpp_volts}')
        self.inst.write('SOURce2:VOLTage:LEVel:IMMediate:OFFSet 0')
        self.inst.write('SOURce2:PHASe 0')                 # 初始相位 0

        # --- 关键步骤：同步相位 ---
        # AFG31052 两个通道是独立的，为了保证扫描图像不乱飘，必须强制对齐相位
        # 很多时候只设频率，两个波形起始点会是随机的
        self.inst.write('SOURce1:PHASe:INITiate') 

    def start_scan(self):
        """开启输出"""
        print("启动振镜扫描输出...")
        self.inst.write('OUTPut1:STATe ON')
        self.inst.write('OUTPut2:STATe ON')

    def stop_scan(self):
        """停止输出"""
        print("停止输出")
        self.inst.write('OUTPut1:STATe OFF')
        self.inst.write('OUTPut2:STATe OFF')

    def close(self):
        self.inst.close()
        self.rm.close()

# ================= 使用示例 =================

if __name__ == "__main__":
    # 请替换为你实际的 USB 地址 (通过 rm.list_resources() 获取)
    # 示例: 'USB0::0x0699::0x0353::C010123::INSTR'
    visa_address = 'USB0::0x0699::0x035E::C018251::INSTR' 
    
    try:
        # 1. 连接设备
        galvo = GalvoScanner(visa_address)
        
        # 2. 设置扫描参数
        # 设定 X轴 500Hz，每帧扫描 50行，电压范围 2Vpp (-1V到+1V)
        # 这样 Y轴(帧率) 就是 10Hz
        galvo.setup_raster_scan(x_freq_hz=500, lines_per_frame=50, vpp_volts=2.0)
        
        # 3. 开始扫描
        galvo.start_scan()
        
        print("正在扫描中... 按 Ctrl+C 停止")
        
        # 让程序保持运行，或者你可以加入其他逻辑
        while True:
            time.sleep(1)
            
    except KeyboardInterrupt:
        print("\n用户中断")
    except Exception as e:
        print(f"发生错误: {e}")
    finally:
        # 确保程序退出时关闭输出，保护振镜
        if 'galvo' in locals():
            galvo.stop_scan()
            galvo.close()

d:\coding\miniconda3\envs\diffusion\Lib\site-packages\pyvisa\ctwrapper\highlevel.py:227: VisaIOWarning: VI_WARN_CONFIG_NLOADED (1073676407): The specified configuration either does not exist or could not be loaded. VISA-specified defaults will be used.
  return self.handle_return_value(session, ret_value)  # type: ignore


已连接到仪器: TEKTRONIX,AFG31252,C018251,SCPI:99.0 FV:1.6.1
正在配置扫描参数...
X轴 (CH1): 500 Hz, 2.0 Vpp
Y轴 (CH2): 10.0000 Hz (帧率), 50 行
启动振镜扫描输出...
正在扫描中... 按 Ctrl+C 停止

用户中断
停止输出


# AFG31052单点位置输出测试

In [6]:
import pyvisa
import time


# ==========================
# 用户需要修改的参数
# ==========================

VISA_ADDRESS = 'USB0::0x0699::0x035E::C018251::INSTR'

# 单点测试电压
# CH1 = X 轴
# CH2 = Y 轴
X_VOLTAGE = 0.10     # 单位 V，安全范围：-1.0 ~ +1.0
Y_VOLTAGE = 0.00     # 单位 V，安全范围：-1.0 ~ +1.0

# 最大允许输出电压
VOLTAGE_LIMIT = 1.0  # 只允许 ±1 V

# 测试持续时间，单位秒
# 例如 10 表示输出 10 秒后自动关闭
# 如果想一直输出，可以改成 None
TEST_DURATION = 10


# ==========================
# 安全控制函数
# ==========================

def check_voltage(v, limit=1.0):
    """
    检查输入电压是否在安全范围内。
    超过 ±limit 直接报错，不允许输出。
    """
    v = float(v)

    if not (-limit <= v <= limit):
        raise ValueError(
            f"危险：输入电压 {v:.6f} V 超出安全范围 "
            f"[-{limit:.3f}, +{limit:.3f}] V，已拒绝输出。"
        )

    return v


def safe_output_off(inst):
    """
    尽可能关闭两个通道输出。
    """
    try:
        inst.write("OUTPut1:STATe OFF")
    except Exception:
        pass

    try:
        inst.write("OUTPut2:STATe OFF")
    except Exception:
        pass


# ==========================
# 主程序
# ==========================

rm = None
inst = None

try:
    # 1. 检查电压安全范围
    x_v = check_voltage(X_VOLTAGE, VOLTAGE_LIMIT)
    y_v = check_voltage(Y_VOLTAGE, VOLTAGE_LIMIT)

    print("电压安全检查通过：")
    print(f"CH1 / X 轴 = {x_v:.6f} V")
    print(f"CH2 / Y 轴 = {y_v:.6f} V")
    print()

    # 2. 连接 VISA
    rm = pyvisa.ResourceManager()

    print("当前可用 VISA 资源：")
    print(rm.list_resources())
    print()

    inst = rm.open_resource(VISA_ADDRESS)
    inst.timeout = 5000

    idn = inst.query("*IDN?").strip()
    print("已连接仪器：")
    print(idn)
    print()

    # 3. 先关闭输出，避免仪器保持上一次状态
    safe_output_off(inst)
    time.sleep(0.2)

    # 4. 清除状态
    inst.write("*CLS")

    # 5. 设置 CH1 为固定 DC 电压
    inst.write("SOURce1:FUNCtion DC")
    inst.write(f"SOURce1:VOLTage:LEVel:IMMediate:OFFSet {x_v:.6f}")

    # 6. 设置 CH2 为固定 DC 电压
    inst.write("SOURce2:FUNCtion DC")
    inst.write(f"SOURce2:VOLTage:LEVel:IMMediate:OFFSet {y_v:.6f}")

    # 7. 尝试设置为高阻负载
    # 如果这条命令你的设备不支持，也不会影响后续输出
    try:
        inst.write("OUTPut1:IMPedance INFinity")
        inst.write("OUTPut2:IMPedance INFinity")
    except Exception:
        print("提示：设置高阻负载命令未成功，继续执行。")

    # 8. 检查仪器错误
    try:
        err = inst.query("SYSTem:ERRor?").strip()
        print("仪器错误状态：", err)
    except Exception:
        print("未能读取仪器错误状态。")

    print()
    print("即将开启输出...")
    time.sleep(0.5)

    # 9. 开启输出
    inst.write("OUTPut1:STATe ON")
    inst.write("OUTPut2:STATe ON")

    print("输出已开启：")
    print(f"CH1 / X 轴 = {x_v:.6f} V")
    print(f"CH2 / Y 轴 = {y_v:.6f} V")

    # 10. 持续输出一段时间后自动关闭
    if TEST_DURATION is not None:
        print(f"将保持输出 {TEST_DURATION} 秒，然后自动关闭。")
        time.sleep(TEST_DURATION)

        safe_output_off(inst)
        print("测试结束，输出已自动关闭。")
    else:
        print("TEST_DURATION = None，输出将保持开启。")
        print("需要停止时，请在新的 cell 中运行：")
        print('inst.write("OUTPut1:STATe OFF")')
        print('inst.write("OUTPut2:STATe OFF")')

except KeyboardInterrupt:
    print("用户中断，正在关闭输出...")
    if inst is not None:
        safe_output_off(inst)

except Exception as e:
    print("发生错误：")
    print(e)

    if inst is not None:
        print("为安全起见，正在关闭输出...")
        safe_output_off(inst)

finally:
    # 如果 TEST_DURATION 不是 None，程序结束后关闭连接
    # 如果 TEST_DURATION = None，为了让你还能在 Jupyter 里继续控制，不主动 close
    if TEST_DURATION is not None:
        if inst is not None:
            inst.close()
        if rm is not None:
            rm.close()
        print("VISA 连接已关闭。")

电压安全检查通过：
CH1 / X 轴 = 0.100000 V
CH2 / Y 轴 = 0.000000 V

当前可用 VISA 资源：
('USB0::0x0699::0x035E::C018251::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR')

已连接仪器：
TEKTRONIX,AFG31252,C018251,SCPI:99.0 FV:1.6.1

仪器错误状态： 0,"No error"

即将开启输出...
输出已开启：
CH1 / X 轴 = 0.100000 V
CH2 / Y 轴 = 0.000000 V
将保持输出 10 秒，然后自动关闭。
测试结束，输出已自动关闭。
VISA 连接已关闭。
